# 04b — Social Media Charts

Publication-ready charts for @unwelcomedata. Altair + vl-convert → PNG.

**Final curated set:**
1. Felony threshold choropleth
2. Felony lookback period choropleth
3. First-time vs repeat offenders — stacked bar
4. DUI enforcement: checkpoints × vehicle impound — bivariate matrix

In [ ]:
import sys, os, io
from pathlib import Path
import importlib
import pandas as pd
import numpy as np
import altair as alt
import duckdb
import geopandas as gpd
import yaml
import vl_convert as vlc
from PIL import Image as PILImage

PROJECT = Path.cwd()
while not (PROJECT / "config.yaml").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent / 'shared'))

import src.viz_social
importlib.reload(src.viz_social)
from src.viz_social import save_social, _preset_size, _draw_watermark
from viz import PALETTE

with open(PROJECT / "config.yaml") as f:
    cfg = yaml.safe_load(f)

df = pd.read_parquet(PROJECT / "export" / "dui_by_state_v4.parquet")
con = duckdb.connect(str(PROJECT / "data" / "project.duckdb"), read_only=True)

def felony_label(row):
    if row['has_felony_dui'] == 0: return 'Always a misdemeanor'
    t = row['felony_dui_threshold']
    if pd.isna(t): return 'Always a misdemeanor'
    t = int(t)
    if t == 2: return 'Felony on 2nd'
    elif t == 3: return 'Felony on 3rd'
    elif t == 4: return 'Felony on 4th'
    return f'Felony on {t}th'

df['felony_at'] = df.apply(felony_label, axis=1)
print(f"Project: {PROJECT.name}, {df.shape[0]} states x {df.shape[1]} cols")

---
## 1. Felony Threshold Map

In [ ]:
# Disable custom theme to prevent color override
alt.themes.enable('none')

# Colors: misdemeanor=light gray, 2nd=coral (strictest), 3rd=teal, 4th=dark blue (most lenient)
felony_colors = {
    'Always a misdemeanor': '#D9D9D9',
    'Felony on 2nd': '#E76F51',
    'Felony on 3rd': '#2A9D8F',
    'Felony on 4th': '#264653',
}
domain = ['Always a misdemeanor', 'Felony on 2nd', 'Felony on 3rd', 'Felony on 4th']
range_ = [felony_colors[k] for k in domain]

w, h = _preset_size('twitter_landscape')
shapefile = PROJECT / 'data' / 'raw' / 'geo' / 'cb_2022_us_state_20m.shp'
geo = gpd.read_file(shapefile)
geo = geo[~geo['STATEFP'].isin({'60','66','69','72','78'})].copy()
map_data = df[['state_fips', 'felony_at']].copy()
map_data['state_fips'] = map_data['state_fips'].astype(str).str.zfill(2)
geo = geo.merge(map_data, left_on='STATEFP', right_on='state_fips', how='left')
geo_json = geo.__geo_interface__
for f in geo_json['features']:
    for k, v in f.get('properties', {}).items(): f[k] = v

map_chart = alt.Chart(alt.Data(values=geo_json['features'])).mark_geoshape(
    stroke='white', strokeWidth=0.8,
).encode(
    color=alt.Color('felony_at:N', scale=alt.Scale(domain=domain, range=range_), legend=None),
    tooltip=[alt.Tooltip('NAME:N', title='State'), alt.Tooltip('felony_at:N', title='Felony threshold')],
).project(type='albersUsa').properties(width=w-350, height=h-180)

count_data = df['felony_at'].value_counts().reindex(domain).reset_index()
count_data.columns = ['category', 'count']

bar_layers, label_layers = [], []
for _, row in count_data.iterrows():
    cat, cnt = row['category'], row['count']
    row_df = pd.DataFrame([{'category': cat, 'count': cnt}])
    bar_layers.append(alt.Chart(row_df).mark_bar(cornerRadiusEnd=3, color=felony_colors[cat]).encode(
        x=alt.X('count:Q', axis=None),
        y=alt.Y('category:N', sort=domain, title='', axis=alt.Axis(labelFontSize=10, labelLimit=180, labelFontWeight='normal'))))
    label_layers.append(alt.Chart(row_df).mark_text(align='left', dx=4, fontSize=12, fontWeight='bold', color=PALETTE['dark']).encode(
        x='count:Q', y=alt.Y('category:N', sort=domain), text=alt.Text('count:Q', format='d')))

legend_chart = alt.layer(*bar_layers, *label_layers).properties(
    width=160, height=h-300,
    title=alt.Title('States per category', fontSize=12, fontWeight='bold', color=PALETTE['dark'], offset=20))

main2 = alt.hconcat(map_chart, legend_chart).resolve_scale(color='independent').properties(
    title=alt.Title('When Does a DUI Become a Felony?', fontSize=20, anchor='start',
        subtitle=['Number of DUI convictions that triggers a felony charge, by state.'],
        subtitleFontSize=13))

source2 = pd.DataFrame([{'text': 'Source: NASID state enforcement laws, corroborated with NCSL | CA is a wobbler (felony at prosecutor discretion on 4th+)', 'x': 0}])
rule2 = alt.Chart(pd.DataFrame([{'x': 0}])).mark_rule(color='#D1D5DB', strokeWidth=1).encode(y=alt.value(0)).properties(width=w-120, height=2)
text2 = alt.Chart(source2).mark_text(align='left', fontSize=9, color=PALETTE['mid']).encode(
    x=alt.X('x:Q', axis=None, scale=alt.Scale(domain=[0,1])), text='text:N').properties(width=w-120, height=20)

chart2 = alt.vconcat(main2, rule2, text2).configure(
    font='Inter, Helvetica Neue, Arial, sans-serif', background='#FFFFFF',
).configure_view(strokeWidth=0).configure_concat(spacing=8)

save_social(chart2, cfg, 'social_map_felony_threshold', preset='twitter_landscape')
alt.themes.enable('unwelcomedata')
from IPython.display import Image, display
display(Image(filename=str(PROJECT / 'outputs' / 'social' / 'social_map_felony_threshold.png')))

---
## 2. Felony Lookback Period Map

In [ ]:
# --- Lookback map + bars (same format as felony threshold chart) ---
# Bin lookback into categories
def lookback_bin(yrs):
    if yrs == 0: return 'No felony law'
    elif yrs <= 7: return 'Short (5–7 yr)'
    elif yrs <= 15: return 'Standard (10–15 yr)'
    else: return 'Lifetime'

df['lookback_cat'] = df['lookback_years'].apply(lookback_bin)

lookback_colors = {
    'No felony law': '#D9D9D9',
    'Short (5–7 yr)': '#E76F51',
    'Standard (10–15 yr)': '#2A9D8F',
    'Lifetime': '#264653',
}
domain3 = ['No felony law', 'Short (5–7 yr)', 'Standard (10–15 yr)', 'Lifetime']
range3 = [lookback_colors[k] for k in domain3]

w, h = _preset_size('twitter_landscape')
shapefile = PROJECT / 'data' / 'raw' / 'geo' / 'cb_2022_us_state_20m.shp'
geo3 = gpd.read_file(shapefile)
geo3 = geo3[~geo3['STATEFP'].isin({'60','66','69','72','78'})].copy()
map_data3 = df[['state_fips', 'lookback_cat']].copy()
map_data3['state_fips'] = map_data3['state_fips'].astype(str).str.zfill(2)
geo3 = geo3.merge(map_data3, left_on='STATEFP', right_on='state_fips', how='left')
geo3_json = geo3.__geo_interface__
for f in geo3_json['features']:
    for k, v in f.get('properties', {}).items(): f[k] = v

map_chart3 = alt.Chart(alt.Data(values=geo3_json['features'])).mark_geoshape(
    stroke='white', strokeWidth=0.8,
).encode(
    color=alt.Color('lookback_cat:N', scale=alt.Scale(domain=domain3, range=range3), legend=None),
    tooltip=[alt.Tooltip('NAME:N', title='State'), alt.Tooltip('lookback_cat:N', title='Lookback period')],
).project(type='albersUsa').properties(width=w-350, height=h-180)

count_data3 = df['lookback_cat'].value_counts().reindex(domain3).reset_index()
count_data3.columns = ['category', 'count']

bar_layers3, label_layers3 = [], []
for _, row in count_data3.iterrows():
    cat, cnt = row['category'], row['count']
    row_df = pd.DataFrame([{'category': cat, 'count': cnt}])
    bar_layers3.append(alt.Chart(row_df).mark_bar(cornerRadiusEnd=3, color=lookback_colors[cat]).encode(
        x=alt.X('count:Q', axis=None),
        y=alt.Y('category:N', sort=domain3, title='', axis=alt.Axis(labelFontSize=10, labelLimit=180, labelFontWeight='normal'))))
    label_layers3.append(alt.Chart(row_df).mark_text(align='left', dx=4, fontSize=12, fontWeight='bold', color=PALETTE['dark']).encode(
        x='count:Q', y=alt.Y('category:N', sort=domain3), text=alt.Text('count:Q', format='d')))

legend_chart3 = alt.layer(*bar_layers3, *label_layers3).properties(
    width=160, height=h-300,
    title=alt.Title('States per category', fontSize=12, fontWeight='bold', color=PALETTE['dark'], offset=20))

main3 = alt.hconcat(map_chart3, legend_chart3).resolve_scale(color='independent').properties(
    title=alt.Title('How Long Is the Window to Trigger a DUI Felony?', fontSize=20, anchor='start',
        subtitle=['Prior convictions only count toward felony escalation within this lookback period.'],
        subtitleFontSize=13))

source3 = pd.DataFrame([{'text': 'Source: State DUI statutes (ailawyer.pro, NASID, WA SB 5032) | 99 = lifetime; DC, MD, NJ have no felony DUI law', 'x': 0}])
rule3 = alt.Chart(pd.DataFrame([{'x': 0}])).mark_rule(color='#D1D5DB', strokeWidth=1).encode(y=alt.value(0)).properties(width=w-120, height=2)
text3 = alt.Chart(source3).mark_text(align='left', fontSize=9, color=PALETTE['mid']).encode(
    x=alt.X('x:Q', axis=None, scale=alt.Scale(domain=[0,1])), text='text:N').properties(width=w-120, height=20)

chart3 = alt.vconcat(main3, rule3, text3).configure(
    font='Inter, Helvetica Neue, Arial, sans-serif', background='#FFFFFF',
).configure_view(strokeWidth=0).configure_concat(spacing=8)

save_social(chart3, cfg, 'social_map_felony_lookback', preset='twitter_landscape')
alt.themes.enable('unwelcomedata')
from IPython.display import Image, display
display(Image(filename=str(PROJECT / 'outputs' / 'social' / 'social_map_felony_lookback.png')))


---
## 3. First-Time vs Repeat Offenders

In [ ]:
# --- Data prep: first-time vs repeat offenders ---
prior = con.sql('SELECT state_fips, impaired_drivers_known_history, impaired_with_prior_dwi FROM fars_prior_dwi_speed').df()
prior['state_fips'] = prior['state_fips'].astype(str).str.zfill(2)
merged = prior.merge(df[['state_fips', 'felony_at']], on='state_fips', how='left')
order = ['Always a misdemeanor', 'Felony on 4th', 'Felony on 3rd', 'Felony on 2nd']
group_agg = merged.groupby('felony_at').agg(
    total_impaired=('impaired_drivers_known_history', 'sum'),
    total_with_prior=('impaired_with_prior_dwi', 'sum')).reindex(order)
group_agg['pct_repeat'] = (group_agg['total_with_prior'] / group_agg['total_impaired'] * 100).round(1)
group_agg['pct_first_time'] = (100 - group_agg['pct_repeat']).round(1)
group_agg['n_states'] = merged.groupby('felony_at')['state_fips'].nunique().reindex(order)
nat_repeat = group_agg['total_with_prior'].sum() / group_agg['total_impaired'].sum() * 100

chart_data = []
for cat in order:
    row = group_agg.loc[cat]
    label = f"{cat} ({int(row['n_states'])} states)"
    chart_data.append({'group': label, 'type': 'No prior DUI conviction', 'pct': row['pct_first_time'], 'sort_order': order.index(cat)})
    chart_data.append({'group': label, 'type': 'Had prior DUI conviction', 'pct': row['pct_repeat'], 'sort_order': order.index(cat)})
bar_df = pd.DataFrame(chart_data)
bar_df = pd.concat([bar_df, pd.DataFrame([
    {'group': 'National (51 states)', 'type': 'No prior DUI conviction', 'pct': round(100 - nat_repeat, 1), 'sort_order': len(order)},
    {'group': 'National (51 states)', 'type': 'Had prior DUI conviction', 'pct': round(nat_repeat, 1), 'sort_order': len(order)},
])], ignore_index=True)
bar_df['label_x'] = bar_df.apply(lambda r: r['pct']/2 if r['type']=='No prior DUI conviction' else (100-r['pct'])+r['pct']/2, axis=1)
bar_df['label_text'] = bar_df['pct'].apply(lambda x: f'{x:.0f}%')

# --- X / Twitter landscape version (1600x900) ---
w_x, h_x = _preset_size('twitter_landscape')
chart_w_x = w_x - 300  # generous bar width for landscape
bar_h_x = h_x - 310    # room for title + subtitle + footer

bars_x = alt.Chart(bar_df).mark_bar(cornerRadiusEnd=3).encode(
    x=alt.X('pct:Q', scale=alt.Scale(domain=[0, 100]), axis=None),
    y=alt.Y('group:N', sort=alt.SortField(field='sort_order', order='ascending'), title=''),
    color=alt.Color('type:N',
        scale=alt.Scale(domain=['No prior DUI conviction', 'Had prior DUI conviction'], range=['#2A9D8F', '#E76F51']),
        legend=alt.Legend(title=None, orient='none', direction='horizontal',
                         labelFontSize=12, symbolSize=140, legendX=chart_w_x - 340, legendY=-30)),
    order=alt.Order('type:N', sort='descending'))

labels_x = alt.Chart(bar_df[bar_df['pct'] > 4]).mark_text(fontSize=14, fontWeight='bold', color='white').encode(
    x='label_x:Q', y=alt.Y('group:N', sort=alt.SortField(field='sort_order', order='ascending')), text='label_text:N')

source1_x = pd.DataFrame([{'text': 'Source: NHTSA FARS 2024 (PREV_DWI field) | MS reports 0% prior DWI (likely undercount)', 'x': 0}])
rule1_x = alt.Chart(pd.DataFrame([{'x': 0}])).mark_rule(color='#D1D5DB', strokeWidth=1).encode(y=alt.value(0)).properties(width=w_x - 120, height=2)
text1_x = alt.Chart(source1_x).mark_text(align='left', fontSize=9, color=PALETTE['mid']).encode(
    x=alt.X('x:Q', axis=None, scale=alt.Scale(domain=[0,1])), text='text:N').properties(width=w_x - 120, height=20)

chart1_x = alt.vconcat(
    (bars_x + labels_x).properties(width=chart_w_x, height=bar_h_x,
        title=alt.Title(
            'Most DUI Deaths Involve Drivers With No Prior DUI Conviction', fontSize=20,
            subtitle='States grouped by felony-charge DUI threshold. No prior conviction \u2260 first time driving impaired.',
            subtitleFontSize=13)),
    rule1_x, text1_x).configure_concat(spacing=8)

save_social(chart1_x, cfg, 'social_first_time_vs_repeat_x', preset='twitter_landscape')
from IPython.display import Image, display
display(Image(filename=str(PROJECT / 'outputs' / 'social' / 'social_first_time_vs_repeat_x.png')))

---
## 4. DUI Enforcement: Checkpoints × Vehicle Impound

In [ ]:
# --- Bivariate matrix: checkpoints × mandatory vehicle impound (Altair + Pillow composite) ---
import io

# Load impound data
df_impound = pd.read_parquet(PROJECT / 'data' / 'interim' / 'vehicle_impound_laws.parquet')
df_impound['has_mandatory_impound'] = ((df_impound['vehicle_impound_law'] == 'yes') & (df_impound['impound_mandatory'] == 'mandatory')).astype(int)

# Merge with enforcement data
df4 = df[['state_fips', 'state_abbr', 'state_name', 'checkpoints_permitted']].copy()
df4 = df4.merge(df_impound[['state_fips', 'has_mandatory_impound']], on='state_fips', how='left')
df4['has_mandatory_impound'] = df4['has_mandatory_impound'].fillna(0).astype(int)

# Assign enforcement quadrants
def impound_quad(row):
    cp = row['checkpoints_permitted'] == 1
    imp = row['has_mandatory_impound'] == 1
    if cp and imp: return 'Both: checkpoints + impound'
    elif cp and not imp: return 'Checkpoints only'
    elif not cp and imp: return 'Impound only (no checkpoints)'
    else: return 'Neither'

df4['enforcement_quad'] = df4.apply(impound_quad, axis=1)

QUAD_COLORS = {
    'Both: checkpoints + impound': '#264653',
    'Impound only (no checkpoints)': '#2A9D8F',
    'Checkpoints only': '#E76F51',
    'Neither': '#D9D9D9',
}

df4['quad_color'] = df4['enforcement_quad'].map(QUAD_COLORS)

shapefile = PROJECT / 'data' / 'raw' / 'geo' / 'cb_2022_us_state_20m.shp'
geo4 = gpd.read_file(shapefile)
geo4 = geo4[~geo4['STATEFP'].isin({'60','66','69','72','78'})].copy()
map_data4 = df4[['state_fips', 'quad_color', 'enforcement_quad']].copy()
map_data4['state_fips'] = map_data4['state_fips'].astype(str).str.zfill(2)
geo4 = geo4.merge(map_data4, left_on='STATEFP', right_on='state_fips', how='left')
geo4['quad_color'] = geo4['quad_color'].fillna('#E5E7EB')
geo4_json = geo4.__geo_interface__
for f in geo4_json['features']:
    for k, v in f.get('properties', {}).items(): f[k] = v

w, h = _preset_size('twitter_landscape')

map_chart4 = alt.Chart(alt.Data(values=geo4_json['features'])).mark_geoshape(
    stroke='white', strokeWidth=0.6,
).encode(
    color=alt.Color('quad_color:N', scale=None),
    tooltip=[alt.Tooltip('NAME:N', title='State'),
             alt.Tooltip('enforcement_quad:N', title='Category')],
).project(type='albersUsa').properties(width=w-60, height=h-120,
    title=alt.Title('Can They Stop You AND Take Your Car?', fontSize=20, anchor='start',
        subtitle=['Checkpoints = sobriety checkpoints legal. Impound = mandatory vehicle seizure upon DUI.',
                  'Dark: both tools. Gray: neither (TX, ID, MT, AK, WY).'],
        subtitleFontSize=13))

source4 = pd.DataFrame([{'text': 'Source: NASID, NHTSA, state statutes | @unwelcomedata', 'x': 0}])
rule4 = alt.Chart(pd.DataFrame([{'x': 0}])).mark_rule(color='#D1D5DB', strokeWidth=1).encode(y=alt.value(0)).properties(width=w-120, height=2)
text4 = alt.Chart(source4).mark_text(align='left', fontSize=9, color=PALETTE['mid']).encode(
    x=alt.X('x:Q', axis=None, scale=alt.Scale(domain=[0,1])), text='text:N').properties(width=w-120, height=20)
chart4_base = alt.vconcat(map_chart4, rule4, text4).configure_concat(spacing=8)

# Render
map4_png = vlc.vegalite_to_png(chart4_base.to_dict(), scale=2.0)
map4_img = PILImage.open(io.BytesIO(map4_png)).resize((w, h), PILImage.LANCZOS)

# Build 2x2 matrix legend in Altair
LEGEND_SIZE = 120
q_counts = df4['enforcement_quad'].value_counts()
legend_data4 = [
    {'x': 0, 'y': 0, 'color': QUAD_COLORS['Neither'], 'label': str(q_counts.get('Neither', 0))},
    {'x': 1, 'y': 0, 'color': QUAD_COLORS['Checkpoints only'], 'label': str(q_counts.get('Checkpoints only', 0))},
    {'x': 0, 'y': 1, 'color': QUAD_COLORS['Impound only (no checkpoints)'], 'label': str(q_counts.get('Impound only (no checkpoints)', 0))},
    {'x': 1, 'y': 1, 'color': QUAD_COLORS['Both: checkpoints + impound'], 'label': str(q_counts.get('Both: checkpoints + impound', 0))},
]
legend_df4 = pd.DataFrame(legend_data4)

legend_rects = alt.Chart(legend_df4).mark_rect(strokeWidth=1.5, stroke='white').encode(
    x=alt.X('x:O', axis=None),
    y=alt.Y('y:O', axis=None, sort='descending'),
    color=alt.Color('color:N', scale=None)
).properties(width=LEGEND_SIZE, height=LEGEND_SIZE)

legend_labels = alt.Chart(legend_df4).mark_text(fontSize=13, fontWeight='bold', color='white').encode(
    x=alt.X('x:O', axis=None),
    y=alt.Y('y:O', axis=None, sort='descending'),
    text='label:N'
).properties(width=LEGEND_SIZE, height=LEGEND_SIZE)

x_lbl = alt.Chart(pd.DataFrame([{'text': 'Checkpoints \u2192'}])).mark_text(
    fontSize=10, color=PALETTE['dark'], fontWeight='bold', align='center', dx=15
).encode(text='text:N').properties(width=LEGEND_SIZE + 16, height=16)

y_lbl = alt.Chart(pd.DataFrame([{'text': 'Vehicle impound \u2192'}])).mark_text(
    fontSize=10, color=PALETTE['dark'], fontWeight='bold', angle=270, align='center'
).encode(text='text:N').properties(width=16, height=LEGEND_SIZE)

lgrid = alt.hconcat(y_lbl, (legend_rects + legend_labels)).resolve_scale(color='independent')
legend_assembled = alt.vconcat(lgrid, x_lbl).resolve_scale(color='independent')

legend4_png = vlc.vegalite_to_png(legend_assembled.to_dict(), scale=2.0)
legend4_img = PILImage.open(io.BytesIO(legend4_png))
sf4 = 180 / legend4_img.width
legend4_img = legend4_img.resize((int(legend4_img.width * sf4), int(legend4_img.height * sf4)), PILImage.LANCZOS)

# Composite legend lower-right
paste_x4, paste_y4 = int(w * 0.88), int(h * 0.55)
map4_img = map4_img.convert('RGBA')
map4_img.paste(legend4_img.convert('RGBA'), (paste_x4, paste_y4), legend4_img.convert('RGBA'))
map4_img = map4_img.convert('RGB')
map4_img = _draw_watermark(map4_img, '@unwelcomedata')

out_path4 = PROJECT / 'outputs' / 'social' / 'social_bivariate_enforcement.png'
map4_img.save(out_path4, format='PNG', optimize=True)
print(f'Saved -> {out_path4} ({w}x{h})')

from IPython.display import Image, display
display(Image(filename=str(out_path4)))


---
## Done

Four charts exported to `outputs/social/`.

In [ ]:
# --- Cleanup ---
con.close()